# Checkpoint 2 — Soluzione (NN) per la stima della zona a conduzione lenta

Dato un evento di 20 segnali unipolari (+ focal points e tempi di stimolazione),
`GroupSolutionCP2` restituisce `[x_SC, y_SC, r_SC]` (centro e raggio della lesione).

**Metodo (due reti pre-addestrate):**
- **Rete A**: 46 feature grezze (tempi di attivazione + focal + t0/t1) → `a_ratio`.
- **Rete B**: heatmap di back-projection (triangolazione delle 2 viste) + contesto
  → `(xc, yc, r)`. Addestrata solo su lesioni osservabili (`|centro| ≤ 1.2`).

In [1]:
import os, sys, time
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

# Path RELATIVI alla cartella del notebook: il metodo funziona su qualsiasi
# macchina (anche quella di valutazione) purche' nella stessa cartella ci siano
# forward_model.py, build_features_B.py e i modelli net_A.pkl, net_B.pkl,
# scaler_reteA.pkl, scaler_XB.pkl.
DATA_DIR  = os.getcwd()
CODE_DIR  = os.getcwd()
OUT_DIR   = os.getcwd()
V_FAST    = 65.541

sys.path.insert(0, CODE_DIR)
import forward_model as fm
from forward_model import extract_times_and_weights, homogeneous_times
import build_features_B as bf
print("Ambiente ok. NumPy:", np.__version__)


Ambiente ok. NumPy: 2.4.4


## 1) Carico dataset

In [ ]:
# === DISABILITATA PER LA CONSEGNA ===
# Caricamento del dataset di TRAINING: non serve all'inferenza sul test set.
# Ri-abilitare solo per riallenare le reti.
# X    = np.load(os.path.join(DATA_DIR, "X_train.npy"))     # grezzo (Rete A)
# X_B  = np.load(os.path.join(DATA_DIR, "X_train_B.npy"))   # heatmap (Rete B)
# y    = np.load(os.path.join(DATA_DIR, "y_train.npy"))
# print(f"X={X.shape}  X_B={X_B.shape}  y={y.shape}")
#
# idx = np.arange(len(X))
# idx_tr, idx_val = train_test_split(idx, test_size=0.10, random_state=42)
# X_tr, X_val   = X[idx_tr],   X[idx_val]
# XB_tr, XB_val = X_B[idx_tr], X_B[idx_val]
# a_tr, a_val   = y[idx_tr,0], y[idx_val,0]
# l_tr, l_val   = y[idx_tr,1:], y[idx_val,1:]
# print(f"Train: {len(idx_tr)}  Val: {len(idx_val)}")


## 2) Rete A — a_ratio

In [ ]:
# === DISABILITATA PER LA CONSEGNA ===
# Training di Rete A (a_ratio): gia' addestrata e salvata in net_A.pkl / scaler_reteA.pkl.
# Ri-abilitare solo per riallenare.
# scaler_X = StandardScaler()
# X_tr_s = scaler_X.fit_transform(X_tr); X_val_s = scaler_X.transform(X_val)
# joblib.dump(scaler_X, os.path.join(OUT_DIR, "scaler_reteA.pkl"))
#
# net_A = MLPRegressor(hidden_layer_sizes=(256,128,64), activation="relu",
#     solver="adam", learning_rate="adaptive", learning_rate_init=1e-3,
#     max_iter=500, early_stopping=True, validation_fraction=0.1,
#     n_iter_no_change=20, random_state=42)
# net_A.fit(X_tr_s, a_tr)
# print(f"Rete A  MAE a_ratio={mean_absolute_error(a_val, net_A.predict(X_val_s)):.4f}")
# joblib.dump(net_A, os.path.join(OUT_DIR, "net_A.pkl"))


## 3) Rete B — lesione (heatmap back-projection)

In [ ]:
# === DISABILITATA PER LA CONSEGNA ===
# Training di Rete B (lesione): gia' addestrata e salvata in net_B.pkl / scaler_XB.pkl.
# Usa il FILTRO OSSERVABILITA' (|centro|<=1.2): le lesioni fuori dalla nuvola dei
# sensori sono non identificabili e, se incluse, spingono la rete verso la media.
# Ri-abilitare solo per riallenare.
# obs_tr = np.hypot(l_tr[:,0], l_tr[:,1]) <= 1.2
# print(f"Rete B addestrata su {obs_tr.sum()}/{len(l_tr)} campioni osservabili (|centro|<=1.2)")
#
# scaler_XB = StandardScaler()
# XB_tr_s = scaler_XB.fit_transform(XB_tr[obs_tr]); XB_val_s = scaler_XB.transform(XB_val)
# joblib.dump(scaler_XB, os.path.join(OUT_DIR, "scaler_XB.pkl"))
#
# t0=time.time()
# net_B = MLPRegressor(hidden_layer_sizes=(256,256,128,64), activation="relu",
#     solver="adam", learning_rate="adaptive", learning_rate_init=1e-3,
#     max_iter=600, early_stopping=True, validation_fraction=0.1,
#     n_iter_no_change=25, random_state=42)
# net_B.fit(XB_tr_s, l_tr[obs_tr])
# l_pred = net_B.predict(XB_val_s)
#
# # valutazione: oltre al MAE globale, riporto la fascia RILEVANTE (|centro|<1.0),
# # l'unica rappresentativa dei 5 scenari reali di test.
# rel = np.hypot(l_val[:,0], l_val[:,1]) < 1.0
# mae = mean_absolute_error(l_val, l_pred, multioutput="raw_values")
# mae_rel = mean_absolute_error(l_val[rel], l_pred[rel], multioutput="raw_values")
# pe_rel = np.hypot(l_pred[rel,0]-l_val[rel,0], l_pred[rel,1]-l_val[rel,1])
# print(f"Rete B: {time.time()-t0:.1f}s")
# print(f"  MAE(tutto)   xc={mae[0]:.4f} yc={mae[1]:.4f} r={mae[2]:.4f}")
# print(f"  MAE(|c|<1.0) xc={mae_rel[0]:.4f} yc={mae_rel[1]:.4f} r={mae_rel[2]:.4f}"
#       f"  | pos_err med={np.median(pe_rel):.3f} mean={pe_rel.mean():.3f}")
# joblib.dump(net_B, os.path.join(OUT_DIR, "net_B.pkl"))


## 4) Valutazione sui 5 scenari reali

In [ ]:
# === DISABILITATA PER LA CONSEGNA ===
# Valutazione diagnostica sui 5 scenari noti: non serve all'inferenza sul test set.
# UP = CODE_DIR
# signals  = np.load(os.path.join(UP, "checkpoint2dataset.npy"))
# aniso    = np.load(os.path.join(UP, "checkpoint2anisotropy.npy")).ravel()
# focal    = np.load(os.path.join(UP, "checkpoint2FocalPoints.npy"))
# starting = np.load(os.path.join(UP, "checkpoint2starting.npy"))
# REF = np.array([[-0.407,0.299,0.578],[0.810,0.113,0.394],[-0.816,-0.686,0.274],
#                 [0.197,-0.242,0.042],[0.647,-0.690,0.244]])
#
# print(f"{'scen':>4} {'a_true':>7} {'a_pred':>7} {'|xc_e|':>8} {'|yc_e|':>8} {'|r_e|':>7}")
# print("-"*48)
# for ev in range(5):
#     (x0,y0),(x1,y1)=focal[ev]; t0,t1=starting[ev]
#     tf0,tf1,_,_=extract_times_and_weights(signals[ev], t0, t1)
#     feat=np.concatenate([tf0,tf1,[x0,y0,x1,y1,t0,t1]]).reshape(1,-1)
#     a_pred=float(np.clip(net_A.predict(scaler_X.transform(feat)).ravel()[0],1.0,9.0))
#
#     fB = bf.build_backproj_features(tf0,tf1,x0,y0,t0,x1,y1,t1,a_pred,v_fast=V_FAST).reshape(1,-1)
#     xc_p,yc_p,r_p = net_B.predict(scaler_XB.transform(fB))[0]
#
#     xc_t,yc_t,r_t=REF[ev]
#     print(f"{ev:>4} {aniso[ev]:>7.1f} {a_pred:>7.3f} "
#           f"{abs(xc_p-xc_t):>8.4f} {abs(yc_p-yc_t):>8.4f} {abs(r_p-r_t):>7.4f}")


## 4b) Visualizzazione: heatmap + lesione vera/stimata

In [ ]:
# === DISABILITATA PER LA CONSEGNA ===
# Visualizzazione diagnostica (heatmap + lesione vera/stimata): non serve all'inferenza.
# import matplotlib.pyplot as plt
#
# def plot_backproj(tf0, tf1, x0, y0, t0, x1, y1, t1, a_ratio,
#                   xc_pred, yc_pred, r_pred,
#                   xc_true=None, yc_true=None, r_true=None, titolo=""):
#     """Mostra la heatmap di back-projection con lesione vera, stimata e sensori."""
#     hom0 = homogeneous_times(x0, y0, t0, a_ratio, v_fast=V_FAST)
#     hom1 = homogeneous_times(x1, y1, t1, a_ratio, v_fast=V_FAST)
#     h0 = bf._backproject_one(x0, y0, tf0 - hom0)
#     h1 = bf._backproject_one(x1, y1, tf1 - hom1)
#     heat = h0 * h1
#     if heat.max() > 0:
#         heat = heat / heat.max()
#
#     fig, ax = plt.subplots(figsize=(7, 7))
#     ext = [bf.DOMAIN[0], bf.DOMAIN[1], bf.DOMAIN[0], bf.DOMAIN[1]]
#     im = ax.imshow(heat, origin='lower', extent=ext, cmap='hot', alpha=0.85)
#     plt.colorbar(im, label='heatmap (prob. lesione)', fraction=0.046)
#
#     ax.scatter(fm.SENSOR_LOCS[:,0], fm.SENSOR_LOCS[:,1], c='cyan', s=40,
#                edgecolors='k', label='sensori', zorder=5)
#     ax.scatter([x0, x1], [y0, y1], c='lime', marker='*', s=300,
#                edgecolors='k', label='focal points', zorder=6)
#
#     th = np.linspace(0, 2*np.pi, 100)
#     ax.plot(xc_pred + r_pred*np.cos(th), yc_pred + r_pred*np.sin(th),
#             'b--', lw=2.5, label='lesione STIMATA')
#     ax.scatter([xc_pred], [yc_pred], c='blue', marker='+', s=150, zorder=6)
#     if xc_true is not None:
#         ax.plot(xc_true + r_true*np.cos(th), yc_true + r_true*np.sin(th),
#                 'g-', lw=2.5, label='lesione VERA')
#         ax.scatter([xc_true], [yc_true], c='green', marker='x', s=100, zorder=6)
#
#     ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect('equal')
#     ax.legend(loc='upper right', fontsize=8)
#     ax.set_title(titolo)
#     plt.tight_layout()
#     plt.show()
#
#
# for ev in range(5):
#     (x0,y0),(x1,y1) = focal[ev]; t0,t1 = starting[ev]
#     tf0,tf1,_,_ = extract_times_and_weights(signals[ev], t0, t1)
#     feat = np.concatenate([tf0,tf1,[x0,y0,x1,y1,t0,t1]]).reshape(1,-1)
#     a_pred = float(np.clip(net_A.predict(scaler_X.transform(feat)).ravel()[0], 1.0, 9.0))
#     fB = bf.build_backproj_features(tf0,tf1,x0,y0,t0,x1,y1,t1,a_pred,v_fast=V_FAST).reshape(1,-1)
#     xc_p,yc_p,r_p = net_B.predict(scaler_XB.transform(fB))[0]
#     xc_t,yc_t,r_t = REF[ev]
#     plot_backproj(tf0,tf1,x0,y0,t0,x1,y1,t1,a_pred,
#                   xc_p,yc_p,r_p, xc_t,yc_t,r_t,
#                   titolo=f"Scenario {ev}  (a_pred={a_pred:.2f})  "
#                          f"err_pos={np.hypot(xc_p-xc_t,yc_p-yc_t):.2f}")


## 5) `GroupSolutionCP2`

In [2]:
def GroupSolutionCP2(signals_event, x_0,y_0,t_0, x_1,y_1,t_1,
                     _A=None,_B=None,_sX=None,_sXB=None):
    if _A is None:
        _A=joblib.load(os.path.join(OUT_DIR,"net_A.pkl"))
        _B=joblib.load(os.path.join(OUT_DIR,"net_B.pkl"))
        _sX=joblib.load(os.path.join(OUT_DIR,"scaler_reteA.pkl"))
        _sXB=joblib.load(os.path.join(OUT_DIR,"scaler_XB.pkl"))
    from forward_model import extract_times_and_weights
    import build_features_B as bf
    tf0,tf1,_,_=extract_times_and_weights(signals_event, t_0, t_1)
    feat=np.concatenate([tf0,tf1,[x_0,y_0,x_1,y_1,t_0,t_1]]).reshape(1,-1)
    a_pred=float(np.clip(_A.predict(_sX.transform(feat)).ravel()[0],1.0,9.0))
    fB=bf.build_backproj_features(tf0,tf1,x_0,y_0,t_0,x_1,y_1,t_1,a_pred,v_fast=V_FAST).reshape(1,-1)
    xc,yc,r=_B.predict(_sXB.transform(fB))[0]
    return [float(xc),float(yc),float(r)]

# --- Chiamata di prova DISABILITATA per la consegna (usava le variabili dei
#     blocchi di training/valutazione, non necessari all'inferenza). ---
# res=GroupSolutionCP2(signals[0],*focal[0][0],starting[0][0],*focal[0][1],starting[0][1],
#                      _A=net_A,_B=net_B,_sX=scaler_X,_sXB=scaler_XB)
# print("Scen 0 pred:", [round(v,4) for v in res], " ref:", REF[0].tolist())


In [ ]:
# === DISABILITATA PER LA CONSEGNA ===
# Analisi diagnostica (de-compressione radiale): non serve all'inferenza.
# import numpy as np
# preds = net_B.predict(XB_val_s)
#
# d_pred = np.sqrt(preds[:,0]**2 + preds[:,1]**2)
# d_true = np.sqrt(l_val[:,0]**2 + l_val[:,1]**2)
#
# k = np.sum(d_pred*d_true)/np.sum(d_pred*d_pred)
# print(f"Fattore di de-compressione radiale k = {k:.3f}")
# print(f"  (d_pred media={d_pred.mean():.3f}, d_true media={d_true.mean():.3f})")
#
# scale = k
# xc_corr = preds[:,0]*scale
# yc_corr = preds[:,1]*scale
# err_orig = np.sqrt((preds[:,0]-l_val[:,0])**2+(preds[:,1]-l_val[:,1])**2).mean()
# err_corr = np.sqrt((xc_corr-l_val[:,0])**2+(yc_corr-l_val[:,1])**2).mean()
# print(f"\nErrore posizione: originale={err_orig:.3f}  dopo de-compressione={err_corr:.3f}")
#
# corr_dist = np.corrcoef(d_pred, d_true)[0,1]
# print(f"corr(d_pred, d_true) = {corr_dist:.3f}  (se alta, la correzione lineare ha senso)")
